# Parallel Processing with Dask

`pygeodata` integrates with [Dask](https://dask.org/) via `build_dask_graph`.
This constructs a lazy computation graph respecting data dependencies,
allowing Dask to parallelise work across cores or a distributed cluster.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

from pygeodata import DataLoader, SpatialSpec, setconfig
from pygeodata.processors.reprojection import Reprojector
from pygeodata.parallel import build_dask_graph
from pyproj import CRS
from affine import Affine
import dask

setconfig(path_data_processed=Path("./data/processed"))

spec = SpatialSpec(
    crs=CRS.from_epsg(4326),
    transform=Affine(0.25, 0, -180, 0, -0.25, 90),
    shape=(720, 1440),
)

## Process multiple years in parallel

In [ ]:
@dataclass
class AnnualNDVILoader(DataLoader):
    year: int = 2020

    @property
    def processor(self):
        return Reprojector(srcpath=f"data/raw/ndvi_{self.year}.tif")


loaders = [AnnualNDVILoader(year=y) for y in range(2000, 2024)]
print(f"Created {len(loaders)} loaders")

In [ ]:
# Build one Dask delayed node per loader
tasks = [build_dask_graph(loader, spec=spec) for loader in loaders]

print("First 3 task keys:")
for t in tasks[:3]:
    print(" ", t.key)

# Execute — only processes loaders that are not already cached
# dask.compute(*tasks)  # uncomment with real data

## Distributed cluster (optional)

Connect a Dask client before calling `compute()` to distribute
across many workers.

In [ ]:
# from dask.distributed import Client
# client = Client(n_workers=8, threads_per_worker=1)
# print("Dashboard:", client.dashboard_link)
#
# dask.compute(*tasks)
# client.close()

# Local thread pool (no cluster needed):
# dask.compute(*tasks, scheduler='threads', num_workers=8)

## Nested DAG: dependencies are wired automatically

Pass the root loader — `build_dask_graph` recursively discovers all
embedded upstream `DataLoader` parameters and wires them as Dask
dependencies in the correct order.

In [ ]:
from pygeodata.processors.rasterizer import Rasterizer
import numpy as np

@dataclass
class ForestMaskLoader(DataLoader):
    @property
    def processor(self):
        return Rasterizer(
            srcpath=Path("data/raw/forest.gpkg"),
            values=1, dtype=np.uint8, fill_value=0,
        )

@dataclass
class MaskedNDVI(DataLoader):
    ndvi: AnnualNDVILoader = None
    mask: ForestMaskLoader = None

    def __post_init__(self):
        self.ndvi = self.ndvi or AnnualNDVILoader(year=2020)
        self.mask = self.mask or ForestMaskLoader()

    def process(self, spec):
        from pygeodata import load
        ndvi = load(self.ndvi, spec)
        mask = load(self.mask, spec)
        ndvi.where(mask == 1).rio.to_raster(self.get_processed_path(spec))


root = MaskedNDVI()
task = build_dask_graph(root, spec=spec)
print("Root task key:", task.key)
# dask.compute(task)  # runs mask → ndvi → masked in correct order